# MobileNetV3-Small Parking Slot Occupancy Training on Google Colab (GPU)

This notebook trains and evaluates the MobileNetV3-small classifier on PKLot and CNRPark datasets using a free Colab GPU.

### Pipeline Steps:
1. Mount Google Drive for persistence.
2. Clone repository & install dependencies.
3. Download dataset.
4. Execute full training with Mixed Precision (AMP) and Early Stopping.
5. Save metrics and checkpoints back to Google Drive.

## 1. Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Clone Repository & Install Requirements

In [ ]:
# Clone or navigate to repo
%cd /content
!if [ ! -d "smart-parking-platform" ]; then git clone -b vision-baseline https://github.com/bh00mi-11/smart-parking-platform.git; fi
%cd /content/smart-parking-platform/vision

!pip install -r requirements.txt

## 3. Download & Prepare Datasets
Downloads full-frame samples, official annotations, CNRPark patches, and verified train/val/test splits.

In [ ]:
!python scripts/download_data.py
!python scripts/inspect_data.py

## 4. Run MobileNetV3-Small Classifier Training (Full GPU Run)
Runs full training using `AdamW`, `CrossEntropyLoss` with class-frequency balancing, and evaluation on the unseen test split.

In [ ]:
# Run full training on GPU (5 to 10 epochs)
!python src/train_classifier.py --epochs 8 --batch-size 32 --lr 3e-4 --split-protocol time-based

## 5. Save Checkpoint & Metrics to Google Drive

In [ ]:
import shutil
drive_save_dir = "/content/drive/MyDrive/smart_parking_vision_outputs"
os.makedirs(drive_save_dir, exist_ok=True)

# Copy model checkpoint, metrics json, and confusion matrix figure
shutil.copy("outputs/mobilenetv3_best.pth", os.path.join(drive_save_dir, "mobilenetv3_best.pth"))
shutil.copy("outputs/metrics.json", os.path.join(drive_save_dir, "metrics.json"))
shutil.copy("outputs/confusion_matrix.png", os.path.join(drive_save_dir, "confusion_matrix.png"))
print(f"Artifacts successfully backed up to: {drive_save_dir}")

## 6. Test Backend classify_slots API Function

In [ ]:
import sys
sys.path.insert(0, ".")
from src.train_classifier import build_model
from src.classify_slots import classify_slots
import glob

model = build_model(num_classes=2)
ckpt = torch.load("outputs/mobilenetv3_best.pth", map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])

sample_images = glob.glob("data/PKLot/**/*.jpg", recursive=True)
if sample_images:
    test_img = sample_images[0]
    test_slots = [
        {"slot_id": "A1", "bbox": [100, 150, 80, 120]},
        {"slot_id": "A2", "rotated_rect": {"cx": 400.0, "cy": 450.0, "w": 85.0, "h": 130.0, "angle": -20.0}}
    ]
    preds = classify_slots(test_img, test_slots, model, device="cuda" if torch.cuda.is_available() else "cpu")
    print("Backend classify_slots API output:", preds)